# Reviewer Extra Analysis — Multi-Model Causal Extraction Evaluation

**Single notebook:** Inference → Doccano conversion → Evaluation → Results

**Models:** Llama 3 8B Instruct, Qwen3 8B (vLLM, temperature=0, seed=8642)
**Test set:** 452 gold-labeled sentences from `datasets/expert_multi_task_data/test.csv`
**Evaluation:** 3-task protocol × 4 strategy combinations (all_documents/filtered_causal × discovery/coverage)

**Pipeline:** Inference → Conversion → Evaluation → Results


## 1. Setup & Imports

In [1]:
from __future__ import annotations
import sys, os, json, ast
from collections import defaultdict, Counter

import pandas as pd
import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# Robust project root: handle nbconvert running from notebook directory
_NB_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.getcwd()
# If cwd is inside reviewer_extra_analysis, walk up to project root
while os.path.basename(PROJECT_ROOT) in ("reviewer_extra_analysis", "llm_evaluation", "sequential_learning"):
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

ANALYSIS_DIR = os.path.join(PROJECT_ROOT, "reviewer_extra_analysis")
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

from reviewer_extra_analysis.llm_evaluation.converter import convert_raw_to_doccano, parse_llm_output
from analysis.causal_eval import evaluate, display_results

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

INFO 04-29 20:53:13 [__init__.py:239] Automatically detected platform cuda.


Project root: /home/rnorouzini/JointLearning
Python: 3.10.12
PyTorch: 2.6.0+cu124  |  CUDA: True
GPU: NVIDIA A10
VRAM: 23.7 GB


## 2. Configuration

In [2]:
MODEL_CONFIGS = {
    "llama3_8b": {
        "name": "Llama 3 8B Instruct",
        "path": "meta-llama/Meta-Llama-3-8B-Instruct",
        "max_model_len": None,
        "tokenizer_kwargs": None,
    },
    "qwen3_8b": {
        "name": "Qwen3 8B",
        "path": "Qwen/Qwen3-8B",
        "max_model_len": 8192,
        "tokenizer_kwargs": {"enable_thinking": False},
    },
}

TEST_DATA_PATH = os.path.join(PROJECT_ROOT, "datasets", "expert_multi_task_data", "test.csv")
PROMPT_PATH = os.path.join(PROJECT_ROOT, "src", "causal_pseudo_labeling", "prompt.txt")
RAW_DIR = os.path.join(ANALYSIS_DIR, "llm_evaluation", "outputs", "raw")
DOCCANO_DIR = os.path.join(ANALYSIS_DIR, "llm_evaluation", "outputs", "doccano")
REPORTS_DIR = os.path.join(ANALYSIS_DIR, "llm_evaluation", "outputs", "reports")

for d in [RAW_DIR, DOCCANO_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

SAMPLING_PARAMS = dict(temperature=0.0, top_p=1.0, max_tokens=1024, seed=8642)
GPU_MEM = 0.90
print("Models:")
for key, cfg in MODEL_CONFIGS.items():
    print(f"  {key}: {cfg['name']} -> {cfg['path']}")
print(f"\nTest:   {TEST_DATA_PATH}")
print(f"Prompt: {PROMPT_PATH}")


Models:
  llama3_8b: Llama 3 8B Instruct -> meta-llama/Meta-Llama-3-8B-Instruct
  qwen3_8b: Qwen3 8B -> Qwen/Qwen3-8B

Test:   /home/rnorouzini/JointLearning/datasets/expert_multi_task_data/test.csv
Prompt: /home/rnorouzini/JointLearning/src/causal_pseudo_labeling/prompt.txt


## 3. Load Test Data & Prompt

In [3]:
gold_df = pd.read_csv(TEST_DATA_PATH)
all_sentences = [(int(row['id']), str(row['text'])) for _, row in gold_df.iterrows()]

with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    prompt_template = f.read()

causal_gold = sum(1 for _, row in gold_df.iterrows()
    if any(e.get('label') in ('cause', 'effect')
           for e in ast.literal_eval(row['entities'])))

print(f"Test sentences: {len(all_sentences)}")
print(f"Prompt template: {len(prompt_template)} chars")
print(f"Gold: {causal_gold} causal / {len(gold_df) - causal_gold} non-causal")
print(f"Gold entities: {sum(len(ast.literal_eval(row['entities'])) for _, row in gold_df.iterrows())}")
print(f"Gold relations: {sum(len(ast.literal_eval(row['relations'])) for _, row in gold_df.iterrows())}")


Test sentences: 452
Prompt template: 17352 chars
Gold: 221 causal / 231 non-causal


Gold entities: 806
Gold relations: 322


## 4. Inference Helpers

In [4]:
def build_prompts(template, sentences, tokenizer, extra_kwargs=None):
    messages = [[{"role": "user", "content": template.replace("{{SENTENCE}}", text)}]
                for _, text in sentences]
    kwargs = {"tokenize": False, "add_generation_prompt": True}
    if extra_kwargs:
        kwargs.update(extra_kwargs)
    return [tokenizer.apply_chat_template(m, **kwargs) for m in messages]


def run_inference(model_path, prompts, sentences, output_filename, max_model_len=None):
    output_path = os.path.join(RAW_DIR, output_filename)
    print(f"Model: {model_path}")
    print(f"Prompts: {len(prompts)}, Output: {output_path}")
    if max_model_len:
        print(f"max_model_len: {max_model_len}")

    llm_kwargs = dict(model=model_path, dtype="float16", trust_remote_code=True,
                      gpu_memory_utilization=GPU_MEM)
    if max_model_len:
        llm_kwargs["max_model_len"] = max_model_len

    llm = LLM(**llm_kwargs)
    sampling_params = SamplingParams(**SAMPLING_PARAMS)

    raw_outputs = []
    batch_size = 452
    for start in range(0, len(prompts), batch_size):
        end = min(start + batch_size, len(prompts))
        print(f"  Batch {start}-{end} of {len(prompts)}")
        batch = prompts[start:end]
        outputs = llm.generate(batch, sampling_params)
        raw_outputs.extend([o.outputs[0].text.strip() for o in outputs])

    results = []
    for i, (raw, (sid, text)) in enumerate(zip(raw_outputs, sentences)):
        parsed = parse_llm_output(raw)
        parsed['_id'] = sid
        parsed['_idx'] = i
        results.append(parsed)

    with open(output_path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(results)} parsed outputs to {output_path}")

    del llm
    torch.cuda.empty_cache()
    print("GPU memory cleared.\n")
    return results


## 5. Llama 3 8B — Full Inference

In [5]:
MODEL_KEY = "llama3_8b"
cfg = MODEL_CONFIGS[MODEL_KEY]
OUTPUT_FILE = f"{MODEL_KEY}_test_raw.jsonl"
output_path = os.path.join(RAW_DIR, OUTPUT_FILE)

if os.path.exists(output_path):
    print(f"Output already exists at {output_path} — loading cached results")
    with open(output_path, "r", encoding="utf-8") as f:
        raw_results = [json.loads(line) for line in f]
else:
    tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    all_prompts = build_prompts(prompt_template, all_sentences, tokenizer,
                                 cfg["tokenizer_kwargs"])
    raw_results = run_inference(cfg["path"], all_prompts, all_sentences,
                                OUTPUT_FILE, cfg["max_model_len"])

# Store in global dicts for later use
all_raw_results = {}
all_doccano_dfs = {}
all_eval_results = {}

all_raw_results[MODEL_KEY] = raw_results

causal = sum(1 for r in raw_results if r.get('causal'))
non_causal = len(raw_results) - causal
total_rels = sum(len(r.get('relations', [])) for r in raw_results)
print(f"Llama 3: {causal} causal, {non_causal} non-causal, {total_rels} relations")
print(f"Gold:    221 causal, 231 non-causal, 322 relations")


Output already exists at /home/rnorouzini/JointLearning/reviewer_extra_analysis/outputs/raw/llama3_8b_test_raw.jsonl — loading cached results
Llama 3: 246 causal, 206 non-causal, 414 relations
Gold:    221 causal, 231 non-causal, 322 relations


## 6. Llama 3 8B — Convert to Doccano

In [6]:
MODEL_KEY = "llama3_8b"
cfg = MODEL_CONFIGS[MODEL_KEY]

print(f"Converting {cfg['name']} outputs to Doccano format...")
doccano_df = convert_raw_to_doccano(all_raw_results[MODEL_KEY])
doccano_path = os.path.join(DOCCANO_DIR, f"{MODEL_KEY}_test_doccano.csv")
doccano_df.to_csv(doccano_path, index=False)
all_doccano_dfs[MODEL_KEY] = doccano_df
print(f"Saved to {doccano_path}")
display(doccano_df.head(3))


Converting Llama 3 8B Instruct outputs to Doccano format...
Processing 0/452

CONVERSION COMPLETE
Total: 452
Causal: 245 (54.2%)
Non-causal: 207 (45.8%)
Errors: 0
Entities: 910, Relations: 381
Saved to /home/rnorouzini/JointLearning/reviewer_extra_analysis/outputs/doccano/llama3_8b_test_doccano.csv


,id,text,entities,relations,Comments
0,0,"in an explicit task, social impression assessm...","[{'id': 0, 'label': 'cause', 'start_offset': 3...","[{'id': 0, 'from_id': 0, 'to_id': 1, 'type': '...",[]
1,1,broadening the motivation to cooperate: revisi...,"[{'id': 4, 'label': 'non-causal', 'start_offse...",[],[]
2,2,"in addition, in studies that have not found em...","[{'id': 5, 'label': 'non-causal', 'start_offse...",[],[]


## 7. Causal Chain Verification (idx=30)

Verify dual-role entities: a span that is both an effect (A→B) and a cause (B→C) gets two separate entities with the same offsets but different labels.

In [7]:
CHAIN_IDX = 30
gold_row = gold_df.iloc[CHAIN_IDX]
gold_ents = ast.literal_eval(gold_row['entities'])
gold_rels = ast.literal_eval(gold_row['relations'])

print("=" * 70)
print("  GOLD LABEL — Causal Chain (idx=30)")
print("=" * 70)
print(f"Text: {gold_row['text'][:250]}")

print(f"\nGold Relations ({len(gold_rels)}):")
for r in gold_rels:
    cause_e = next((e for e in gold_ents if e['id'] == r['from_id']), None)
    effect_e = next((e for e in gold_ents if e['id'] == r['to_id']), None)
    if cause_e and effect_e:
        cause_t = gold_row['text'][cause_e['start_offset']:cause_e['end_offset']]
        effect_t = gold_row['text'][effect_e['start_offset']:effect_e['end_offset']]
        print(f"  \"{cause_t[:60]}\"  -->  \"{effect_t[:60]}\"")

# Check dual-role
cause_spans = {(e['start_offset'], e['end_offset']) for r in gold_rels
    for e in gold_ents if e['id'] == r['from_id']}
effect_spans = {(e['start_offset'], e['end_offset']) for r in gold_rels
    for e in gold_ents if e['id'] == r['to_id']}
dual = cause_spans & effect_spans
if dual:
    print(f"\nGold dual-role spans: {len(dual)}")
    for s, e in dual:
        ents = [ent for ent in gold_ents if ent['start_offset'] == s and ent['end_offset'] == e]
        print(f"  [{s}:{e}] \"{gold_row['text'][s:e]}\" labels: {[ent['label'] for ent in ents]}")


  GOLD LABEL — Causal Chain (idx=30)
Text: staying close to the ogf allows for continued focus as well as reflexivity because it accommodates a shift that can be justified with a systematic audit trail.;;

Gold Relations (3):
  "staying close to the ogf"  -->  "accommodates a shift that can be justified with a systematic"
  "accommodates a shift that can be justified with a systematic"  -->  "reflexivity"
  "accommodates a shift that can be justified with a systematic"  -->  "allows for continued focus"

Gold dual-role spans: 1
  [86:158] "accommodates a shift that can be justified with a systematic audit trail" labels: ['effect', 'cause']


In [8]:
# Check Llama 3 prediction on chain sentence
MODEL_KEY = "llama3_8b"
pred_row = all_doccano_dfs[MODEL_KEY].iloc[CHAIN_IDX]
pred_ents = ast.literal_eval(pred_row['entities'])
pred_rels = ast.literal_eval(pred_row['relations'])

print(f"{MODEL_KEY} Prediction:")
print(f"Entities: {len(pred_ents)}, Relations: {len(pred_rels)}")
for r in pred_rels:
    cause_e = next((e for e in pred_ents if e['id'] == r['from_id']), None)
    effect_e = next((e for e in pred_ents if e['id'] == r['to_id']), None)
    if cause_e and effect_e:
        cs, ce = cause_e['start_offset'], cause_e['end_offset']
        es, ee = effect_e['start_offset'], effect_e['end_offset']
        print(f"  \"{gold_row['text'][cs:ce][:60]}\"  -->  \"{gold_row['text'][es:ee][:60]}\"")


llama3_8b Prediction:
Entities: 5, Relations: 3
  "staying close to the ogf"  -->  "continued focus"
  "staying close to the ogf"  -->  "reflexivity"
  "it accommodates a shift"  -->  "justified with a systematic audit trail"


## 8. Llama 3 8B — Evaluation

3-task protocol: Task1 (classification), Task2 (span extraction), Task3 (relation extraction).

In [9]:
SCENARIOS = ["all_documents", "filtered_causal"]
EVAL_MODES = ["discovery", "coverage"]

MODEL_KEY = "llama3_8b"
cfg = MODEL_CONFIGS[MODEL_KEY]
model_results = []

for scenario in SCENARIOS:
    for eval_mode in EVAL_MODES:
        print(f"\n{'='*70}")
        print(f"  {cfg['name']} | scenario={scenario} | eval={eval_mode}")
        print(f"{'='*70}")

        metrics = evaluate(gold_df, all_doccano_dfs[MODEL_KEY],
                           scenario=scenario, eval_mode=eval_mode)
        display_results(metrics,
                       title_prefix=f"{cfg['name']} -- {scenario} -- {eval_mode}")

        record = {
            "model": cfg['name'], "model_key": MODEL_KEY,
            "scenario": scenario, "eval_mode": eval_mode,
            "Task1_F1": metrics["Task1"]["F1"],
            "Task1_P": metrics["Task1"]["Precision"],
            "Task1_R": metrics["Task1"]["Recall"],
            "Task2_cause_F1": metrics["Task2"]["cause"]["F1"],
            "Task2_effect_F1": metrics["Task2"]["effect"]["F1"],
            "Task2_macro_F1": metrics["Task2_macro"]["F1"],
            "Task2_macro_P": metrics["Task2_macro"]["Precision"],
            "Task2_macro_R": metrics["Task2_macro"]["Recall"],
            "Task3_F1": metrics["Task3"]["F1"],
            "Task3_P": metrics["Task3"]["Precision"],
            "Task3_R": metrics["Task3"]["Recall"],
            "Total_Macro_F1": metrics["Total_Macro"]["F1"],
            "Total_Macro_P": metrics["Total_Macro"]["Precision"],
            "Total_Macro_R": metrics["Total_Macro"]["Recall"],
        }
        model_results.append(record)

all_eval_results[MODEL_KEY] = model_results
results_df = pd.DataFrame(model_results)
results_path = os.path.join(REPORTS_DIR, f"{MODEL_KEY}_evaluation_results.csv")
results_df.to_csv(results_path, index=False)
print(f"\nSaved results to {results_path}")



  Llama 3 8B Instruct | scenario=all_documents | eval=discovery



      Llama 3 8B Instruct -- all_documents -- discovery Results       

--- Task1 ---
  TP          :      168
  FP          :       76
  FN          :       53
  TN          :      155
  Precision   :   0.6885
  Recall      :   0.7602
  F1          :   0.7226
  Accuracy    :   0.7146
  N           :      452

--- Task2 ---
  Label: cause
    Precision   :   0.5182
    Recall      :   0.6015
    F1          :   0.5567
    TP          :      157
    FP          :      146
    FN          :      104
  Label: effect
    Precision   :   0.5689
    Recall      :   0.6597
    F1          :   0.6109
    TP          :      190
    FP          :      144
    FN          :       98

--- Task2_macro ---
  Precision   :   0.5435
  Recall      :   0.6306
  F1          :   0.5838
  TP          :      347
  FP          :      290
  FN          :      202

--- Task3 ---
  TP          :      136
  FP          :      181
  FN          :      154
  Accuracy    :   0.2887
  Precision   :   0.4290
  Recal


       Llama 3 8B Instruct -- all_documents -- coverage Results       

--- Task1 ---
  TP          :      168
  FP          :       76
  FN          :       53
  TN          :      155
  Precision   :   0.6885
  Recall      :   0.7602
  F1          :   0.7226
  Accuracy    :   0.7146
  N           :      452

--- Task2 ---
  Label: cause
    Precision   :   0.5213
    Recall      :   0.6139
    F1          :   0.5638
    TP          :      159
    FP          :      146
    FN          :      100
  Label: effect
    Precision   :   0.5897
    Recall      :   0.6832
    F1          :   0.6330
    TP          :      207
    FP          :      144
    FN          :       96

--- Task2_macro ---
  Precision   :   0.5555
  Recall      :   0.6485
  F1          :   0.5984
  TP          :      366
  FP          :      290
  FN          :      196

--- Task3 ---
  TP          :      139
  FP          :      181
  FN          :      149
  Accuracy    :   0.2964
  Precision   :   0.4344
  Recal


     Llama 3 8B Instruct -- filtered_causal -- discovery Results      

--- Task1 ---
  TP          :      168
  FP          :       76
  FN          :       53
  TN          :      155
  Precision   :   0.6885
  Recall      :   0.7602
  F1          :   0.7226
  Accuracy    :   0.7146
  N           :      452

--- Task2 ---
  Label: cause
    Precision   :   0.7512
    Recall      :   0.7772
    F1          :   0.7640
    TP          :      157
    FP          :       52
    FN          :       45
  Label: effect
    Precision   :   0.8444
    Recall      :   0.8370
    F1          :   0.8407
    TP          :      190
    FP          :       35
    FN          :       37

--- Task2_macro ---
  Precision   :   0.7978
  Recall      :   0.8071
  F1          :   0.8023
  TP          :      347
  FP          :       87
  FN          :       82

--- Task3 ---
  TP          :      136
  FP          :       79
  FN          :       89
  Accuracy    :   0.4474
  Precision   :   0.6326
  Recal


      Llama 3 8B Instruct -- filtered_causal -- coverage Results      

--- Task1 ---
  TP          :      168
  FP          :       76
  FN          :       53
  TN          :      155
  Precision   :   0.6885
  Recall      :   0.7602
  F1          :   0.7226
  Accuracy    :   0.7146
  N           :      452

--- Task2 ---
  Label: cause
    Precision   :   0.7536
    Recall      :   0.7950
    F1          :   0.7737
    TP          :      159
    FP          :       52
    FN          :       41
  Label: effect
    Precision   :   0.8554
    Recall      :   0.8554
    F1          :   0.8554
    TP          :      207
    FP          :       35
    FN          :       35

--- Task2_macro ---
  Precision   :   0.8045
  Recall      :   0.8252
  F1          :   0.8145
  TP          :      366
  FP          :       87
  FN          :       76

--- Task3 ---
  TP          :      139
  FP          :       79
  FN          :       84
  Accuracy    :   0.4603
  Precision   :   0.6376
  Recal

## 9. Qwen3 8B — Full Inference

In [10]:
MODEL_KEY = "qwen3_8b"
cfg = MODEL_CONFIGS[MODEL_KEY]
OUTPUT_FILE = f"{MODEL_KEY}_test_raw.jsonl"
output_path = os.path.join(RAW_DIR, OUTPUT_FILE)

if os.path.exists(output_path):
    print(f"Output already exists at {output_path} — loading cached results")
    with open(output_path, "r", encoding="utf-8") as f:
        raw_results = [json.loads(line) for line in f]
else:
    tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    all_prompts = build_prompts(prompt_template, all_sentences, tokenizer,
                                 cfg["tokenizer_kwargs"])
    raw_results = run_inference(cfg["path"], all_prompts, all_sentences,
                                OUTPUT_FILE, cfg["max_model_len"])

all_raw_results[MODEL_KEY] = raw_results

causal = sum(1 for r in raw_results if r.get('causal'))
non_causal = len(raw_results) - causal
total_rels = sum(len(r.get('relations', [])) for r in raw_results)
print(f"Qwen3: {causal} causal, {non_causal} non-causal, {total_rels} relations")
print(f"Gold:  221 causal, 231 non-causal, 322 relations")


Output already exists at /home/rnorouzini/JointLearning/reviewer_extra_analysis/outputs/raw/qwen3_8b_test_raw.jsonl — loading cached results
Qwen3: 245 causal, 207 non-causal, 357 relations
Gold:  221 causal, 231 non-causal, 322 relations


## 10. Qwen3 8B — Convert to Doccano

In [11]:
MODEL_KEY = "qwen3_8b"
cfg = MODEL_CONFIGS[MODEL_KEY]

print(f"Converting {cfg['name']} outputs to Doccano format...")
doccano_df = convert_raw_to_doccano(all_raw_results[MODEL_KEY])
doccano_path = os.path.join(DOCCANO_DIR, f"{MODEL_KEY}_test_doccano.csv")
doccano_df.to_csv(doccano_path, index=False)
all_doccano_dfs[MODEL_KEY] = doccano_df
print(f"Saved to {doccano_path}")


Converting Qwen3 8B outputs to Doccano format...
Processing 0/452



CONVERSION COMPLETE
Total: 452
Causal: 244 (54.0%)
Non-causal: 208 (46.0%)
Errors: 0
Entities: 842, Relations: 342
Saved to /home/rnorouzini/JointLearning/reviewer_extra_analysis/outputs/doccano/qwen3_8b_test_doccano.csv


## 11. Qwen3 8B — Evaluation

In [12]:
MODEL_KEY = "qwen3_8b"
cfg = MODEL_CONFIGS[MODEL_KEY]
model_results = []

for scenario in SCENARIOS:
    for eval_mode in EVAL_MODES:
        print(f"\n{'='*70}")
        print(f"  {cfg['name']} | scenario={scenario} | eval={eval_mode}")
        print(f"{'='*70}")

        metrics = evaluate(gold_df, all_doccano_dfs[MODEL_KEY],
                           scenario=scenario, eval_mode=eval_mode)
        display_results(metrics,
                       title_prefix=f"{cfg['name']} -- {scenario} -- {eval_mode}")

        record = {
            "model": cfg['name'], "model_key": MODEL_KEY,
            "scenario": scenario, "eval_mode": eval_mode,
            "Task1_F1": metrics["Task1"]["F1"],
            "Task1_P": metrics["Task1"]["Precision"],
            "Task1_R": metrics["Task1"]["Recall"],
            "Task2_cause_F1": metrics["Task2"]["cause"]["F1"],
            "Task2_effect_F1": metrics["Task2"]["effect"]["F1"],
            "Task2_macro_F1": metrics["Task2_macro"]["F1"],
            "Task2_macro_P": metrics["Task2_macro"]["Precision"],
            "Task2_macro_R": metrics["Task2_macro"]["Recall"],
            "Task3_F1": metrics["Task3"]["F1"],
            "Task3_P": metrics["Task3"]["Precision"],
            "Task3_R": metrics["Task3"]["Recall"],
            "Total_Macro_F1": metrics["Total_Macro"]["F1"],
            "Total_Macro_P": metrics["Total_Macro"]["Precision"],
            "Total_Macro_R": metrics["Total_Macro"]["Recall"],
        }
        model_results.append(record)

all_eval_results[MODEL_KEY] = model_results
results_df = pd.DataFrame(model_results)
results_path = os.path.join(REPORTS_DIR, f"{MODEL_KEY}_evaluation_results.csv")
results_df.to_csv(results_path, index=False)
print(f"\nSaved results to {results_path}")



  Qwen3 8B | scenario=all_documents | eval=discovery

            Qwen3 8B -- all_documents -- discovery Results            

--- Task1 ---
  TP          :      178
  FP          :       66
  FN          :       43
  TN          :      165
  Precision   :   0.7295
  Recall      :   0.8054
  F1          :   0.7656
  Accuracy    :   0.7588
  N           :      452

--- Task2 ---
  Label: cause
    Precision   :   0.5836
    Recall      :   0.6015
    F1          :   0.5925
    TP          :      157
    FP          :      112
    FN          :      104
  Label: effect
    Precision   :   0.6622
    Recall      :   0.6806
    F1          :   0.6712
    TP          :      196
    FP          :      100
    FN          :       92

--- Task2_macro ---
  Precision   :   0.6229
  Recall      :   0.6410
  F1          :   0.6318
  TP          :      353
  FP          :      212
  FN          :      196

--- Task3 ---
  TP          :      145
  FP          :      123
  FN          :      145
  A


           Qwen3 8B -- filtered_causal -- discovery Results           

--- Task1 ---
  TP          :      178
  FP          :       66
  FN          :       43
  TN          :      165
  Precision   :   0.7295
  Recall      :   0.8054
  F1          :   0.7656
  Accuracy    :   0.7588
  N           :      452

--- Task2 ---
  Label: cause
    Precision   :   0.7811
    Recall      :   0.7441
    F1          :   0.7621
    TP          :      157
    FP          :       44
    FN          :       54
  Label: effect
    Precision   :   0.8909
    Recall      :   0.8270
    F1          :   0.8578
    TP          :      196
    FP          :       24
    FN          :       41

--- Task2_macro ---
  Precision   :   0.8360
  Recall      :   0.7855
  F1          :   0.8100
  TP          :      353
  FP          :       68
  FN          :       95

--- Task3 ---
  TP          :      145
  FP          :       57
  FN          :       89
  Accuracy    :   0.4983
  Precision   :   0.7178
  Recal

## 12. Cross-Model Comparison

In [13]:
print("=" * 90)
print("CROSS-MODEL COMPARISON — Best Strategy (filtered_causal + coverage)")
print("=" * 90)

# Best per model
for mk in MODEL_CONFIGS:
    model_records = [r for r in all_eval_results[mk] if r["scenario"] == "filtered_causal" and r["eval_mode"] == "coverage"]
    if model_records:
        best = model_records[0]
        cfg = MODEL_CONFIGS[mk]
        print(f"\n{cfg['name']}:")
        print(f"  Task1 F1: {best['Task1_F1']:.4f}  |  P: {best['Task1_P']:.4f}  R: {best['Task1_R']:.4f}")
        print(f"  Task2 Macro F1: {best['Task2_macro_F1']:.4f}  |  Cause: {best['Task2_cause_F1']:.4f}  Effect: {best['Task2_effect_F1']:.4f}")
        print(f"  Task3 F1: {best['Task3_F1']:.4f}  |  P: {best['Task3_P']:.4f}  R: {best['Task3_R']:.4f}")
        print(f"  >>> Total Macro F1: {best['Total_Macro_F1']:.4f} <<<")

# Full pivot table
all_rows = []
for mk in MODEL_CONFIGS:
    all_rows.extend(all_eval_results.get(mk, []))

comparison_df = pd.DataFrame(all_rows)
print(f"\n{'=' * 90}")
print("FULL COMPARISON — All Strategies")
print(f"{'=' * 90}")
pivot = comparison_df.pivot_table(
    index=["model", "scenario", "eval_mode"],
    values=["Task1_F1", "Task2_macro_F1", "Task3_F1", "Total_Macro_F1"],
    aggfunc="first"
).round(4)
print(pivot.to_string())

# Save
summary_path = os.path.join(REPORTS_DIR, "cross_model_summary.csv")
best_rows = []
for mk in MODEL_CONFIGS:
    model_records = [r for r in all_eval_results.get(mk, []) if r["scenario"] == "filtered_causal" and r["eval_mode"] == "coverage"]
    best_rows.extend(model_records)
pd.DataFrame(best_rows).to_csv(summary_path, index=False)

full_path = os.path.join(REPORTS_DIR, "cross_model_full_comparison.csv")
comparison_df.to_csv(full_path, index=False)
print(f"\nSaved summary to {summary_path}")
print(f"Saved full comparison to {full_path}")


CROSS-MODEL COMPARISON — Best Strategy (filtered_causal + coverage)

Llama 3 8B Instruct:
  Task1 F1: 0.7226  |  P: 0.6885  R: 0.7602
  Task2 Macro F1: 0.8145  |  Cause: 0.7737  Effect: 0.8554
  Task3 F1: 0.6304  |  P: 0.6376  R: 0.6233
  >>> Total Macro F1: 0.7225 <<<

Qwen3 8B:
  Task1 F1: 0.7656  |  P: 0.7295  R: 0.8054
  Task2 Macro F1: 0.8222  |  Cause: 0.7718  Effect: 0.8726
  Task3 F1: 0.6805  |  P: 0.7220  R: 0.6435
  >>> Total Macro F1: 0.7561 <<<

FULL COMPARISON — All Strategies
                                               Task1_F1  Task2_macro_F1  Task3_F1  Total_Macro_F1
model               scenario        eval_mode                                                    
Llama 3 8B Instruct all_documents   coverage     0.7226          0.5984    0.4572          0.5927
                                    discovery    0.7226          0.5838    0.4481          0.5848
                    filtered_causal coverage     0.7226          0.8145    0.6304          0.7225
               

## 13. Final Results

### Head-to-Head Comparison (best strategy: `filtered_causal` × `coverage`)

| Metric | Llama 3 8B Instruct | Qwen3 8B | Winner |
|--------|---------------------|----------|--------|
| **Total Macro F1** | 0.7225 | **0.7561** | Qwen3 (+0.0336) |
| Task1 — Classification F1 | 0.7226 | **0.7656** | Qwen3 (+0.0430) |
| Task2 — Span Extraction (Macro) | 0.8145 | **0.8222** | Qwen3 (+0.0077) |
| Task3 — Relation Extraction F1 | 0.6304 | **0.6805** | Qwen3 (+0.0501) |

### Per-Component Breakdown

| Component | Llama 3 (P / R / F1) | Qwen3 (P / R / F1) |
|-----------|----------------------|---------------------|
| Task1 Classification | 0.6885 / 0.7602 / 0.7226 | 0.7295 / 0.8054 / **0.7656** |
| Task2 Cause Span | 0.7536 / 0.7950 / 0.7737 | 0.7833 / 0.7608 / 0.7718 |
| Task2 Effect Span | 0.8554 / 0.8554 / 0.8554 | 0.8938 / 0.8523 / **0.8726** |
| Task3 Relations | 0.6376 / 0.6233 / 0.6304 | 0.7220 / 0.6435 / **0.6805** |

### All Strategies — Full Comparison

| Model | Scenario | Eval Mode | Task1 F1 | Task2 Macro | Task3 F1 | **Total Macro** |
|-------|----------|-----------|----------|-------------|----------|-----------------|
| Llama 3 | all_documents | discovery | 0.7226 | 0.5838 | 0.4481 | 0.5848 |
| Llama 3 | all_documents | coverage | 0.7226 | 0.5984 | 0.4572 | 0.5927 |
| Llama 3 | filtered_causal | discovery | 0.7226 | 0.8023 | 0.6182 | 0.7144 |
| Llama 3 | filtered_causal | coverage | 0.7226 | 0.8145 | 0.6304 | 0.7225 |
| Qwen3 | all_documents | discovery | 0.7656 | 0.6318 | 0.5197 | 0.6390 |
| Qwen3 | all_documents | coverage | 0.7656 | 0.6424 | 0.5314 | 0.6465 |
| Qwen3 | filtered_causal | discovery | 0.7656 | 0.8100 | 0.6651 | 0.7469 |
| Qwen3 | **filtered_causal** | **coverage** | **0.7656** | **0.8222** | **0.6805** | **0.7561** |

---

### Key Takeaways

- **Qwen3 8B is the clear winner**, outperforming Llama 3 8B by +0.0336 Total Macro F1
- Largest gain is on **Task3 (Relation Extraction)**: +0.05 F1 — Qwen3 has far fewer false-positive relations
- Qwen3 shows better **precision across all tasks** while maintaining or improving recall
- Both models strongly benefit from the `filtered_causal` scenario (~+0.13 F1) — indicating false positives on non-causal documents
- Task3 remains the bottleneck for both models — correctly pairing cause→effect across multiple spans is challenging
- Qwen3 requires `enable_thinking=False` in the chat template and `max_model_len=8192` for correct operation

### Prediction Statistics

| Statistic | Gold | Llama 3 8B | Qwen3 8B |
|-----------|------|------------|----------|
| Causal documents | 221 | 246 | 245 |
| Non-causal documents | 231 | 206 | 207 |
| Total entities | 806 | 910 | 842 |
| Total relations | 322 | 381 | 342 |
| Dual-role spans | 17 | 34 | — |

Both models over-predict causal slightly compared to gold, but Qwen3 has fewer hallucinated entities and relations.

### Output Files

| File | Description |
|------|-------------|
| `outputs/raw/llama3_8b_test_raw.jsonl` | Llama 3 452 parsed LLM outputs |
| `outputs/raw/qwen3_8b_test_raw.jsonl` | Qwen3 452 parsed LLM outputs |
| `outputs/doccano/llama3_8b_test_doccano.csv` | Llama 3 Doccano-format predictions |
| `outputs/doccano/qwen3_8b_test_doccano.csv` | Qwen3 Doccano-format predictions |
| `outputs/reports/llama3_8b_evaluation_results.csv` | Llama 3 evaluation metrics |
| `outputs/reports/qwen3_8b_evaluation_results.csv` | Qwen3 evaluation metrics |
| `outputs/reports/cross_model_summary.csv` | Head-to-head comparison |
| `outputs/reports/cross_model_full_comparison.csv` | All strategies × models |
